In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
daymet_local = pd.read_parquet("../phenocam_site_in_usa_climate.parquet")
daymet_post  = pd.read_parquet("../post_climate_data.parquet")

daymet_post = (
    daymet_post
    .assign(
        year = lambda x: np.where(x.yday < 1, x.year-1, x.year),
        yday = lambda x: x.yday % 366
    )
    .query(
        'yday != 0' # drop leap days
    )
    .sort_values(["year", "yday"])
    .set_index(["site", "year", "yday"])
)

daymet_local = daymet_local.set_index("site", append=True)

In [ ]:
# From
# https://github.com/akpost21/Spring_pheno_grassland/blob/b5af2ac8f18ce2e5b4415dc27b5becf10ad71bbc/Data%20formatting/Replace_Daymet_precip.R

sites_with_substituted_prcp = [
    "segaarboretummeadow",
    "segabluechute",
    "segahartprairie",
    "segawhitepockets",
    "segablackpoint",
    "kendall",
    "NEON.D13.MOAB.DP1.00033",
    "sevilletanewgrass",
    "srm"
]

In [ ]:
daymet_merge = pd.merge(
    daymet_post[["Tmini", "Tmaxi", "Pi"]],
    daymet_local[["tmin", "tmax", "prcp", "src"]],
    left_index=True,
    right_index=True
)

In [ ]:
daymet_merge

In [ ]:
daymet_merge[
    (daymet_merge.Tmini != daymet_merge.tmin) &
    (~daymet_merge.index.get_level_values("site").isin(sites_with_substituted_prcp)) &
    (daymet_merge.src == "daymet")
]

In [ ]:
daymet_merge.loc["hartprairie", 2016][["Tmaxi", "tmax"]].plot()

In [ ]:
pheno_post  = pd.read_parquet("../post_pheno_data.parquet")
pheno_local = (
    pd.read_parquet("../phenocam_phenology_model.parquet")
        .query("direction == 'rising'")
)

In [ ]:
pheno_local.columns

In [ ]:
pheno_merge = pd.merge(
    pheno_local[["site", "doy_transition_50", "transition_year"]],
    pheno_post,
    right_on=["site", "year"],
    left_on=["site", "transition_year"]
).assign(
    transition_diff = lambda x: x.doy_transition_50 - x.transition_doy
)

In [ ]:
pheno_merge[pheno_merge.transition_diff.abs() > 10]

In [ ]:
(pheno_merge.doy_transition_50 - pheno_merge.transition_doy).hist()